In [24]:
import requests
import uuid

In [126]:
def mcp_request(url: str, method: str, payload_params: dict = None, session_id: str = None, is_notification: bool = False):
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if session_id:
        headers["mcp-session-id"] = session_id

    payload = {
        "jsonrpc": "2.0",
        "method": method,
    }
    if payload_params is not None:
        payload["params"] = payload_params
    if not is_notification:
        payload["id"] = "1"  # 只有请求带 id，通知不带

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=5)
        return response
    except Exception as e:
        return None

In [135]:
url = "http://127.0.0.1:8000/mcp"
# 1) initialize（无会话头）
init_params = {
    "protocolVersion": "2025-03-26",
    "capabilities": {},
    "clientInfo": {"name": "mcp-gateway-test", "version": "0.1.0"}
}
init_resp = mcp_request(url, "initialize", payload_params=init_params)
session_id = init_resp.headers.get("mcp-session-id")

# 2) 发送 notifications/initialized（通知：不带 id，不需要 params）
mcp_request(url, "notifications/initialized", session_id=session_id, is_notification=True)

# 3) 后续请求（携带会话头）
resp = mcp_request(url, "tools/list", session_id=session_id)

# 4) 调用工具
params = {
    "name": "greet",
    "arguments": {
        "name": "test"
    }
}
resp = mcp_request(url, "tools/call", session_id=session_id, payload_params=params)

resp.json()

{'jsonrpc': '2.0',
 'id': '1',
 'result': {'content': [{'type': 'text', 'text': 'Hello, test!'}],
  'structuredContent': {'result': 'Hello, test!'},
  'isError': False}}

In [128]:
init_resp.text, second_resp.text, resp.text

('{"jsonrpc":"2.0","id":"1","result":{"protocolVersion":"2025-03-26","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"StatefulServer","version":"1.13.1"}}}',
 '',
 '{"jsonrpc":"2.0","id":"1","result":{}}')

In [99]:
second_resp.text

'{"jsonrpc":"2.0","id":"1","error":{"code":-32602,"message":"Invalid request parameters","data":""}}'

In [82]:
response.url, response.headers

('http://127.0.0.1:8000/mcp',
 {'date': 'Sat, 06 Sep 2025 13:12:55 GMT', 'server': 'uvicorn', 'content-type': 'application/json', 'mcp-session-id': 'ee8910a289cc4d54832fa51ba1956be7', 'content-length': '99'})

In [100]:
resp.text

'{"jsonrpc":"2.0","id":"1","error":{"code":-32602,"message":"Invalid request parameters","data":""}}'

In [ ]:
from mcp.server.fastmcp import FastMCP
import mcp.types as types

mcp = FastMCP("StatefulServer", json_response=True)   # 默认 stateless_http=False

@mcp.tool()
def greet(name: str = "World") -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")   # 有状态，但客户端每次换新 ID 即可